In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.gold")

In [0]:
bucket = "s3://ro-company-lake"
onrc_firme_path = f"{bucket}/raw_v2/onrc/firme/"

display(dbutils.fs.ls(onrc_firme_path))

In [0]:
od_firme = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("sep", "^")
    .option("encoding", "UTF-8")
    .csv(f"{onrc_firme_path}snapshot_date=*/package=*/od_firme.csv")
)

display(od_firme.limit(20))
print("Rows:", od_firme.count())
print("Columns:", od_firme.columns)

In [0]:
(
    od_firme
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.bronze.onrc_firme_raw")
)

In [0]:
display(spark.sql("""
SELECT COUNT(*) AS rows
FROM company_ro.bronze.onrc_firme_raw
"""))